Direct YML Files analysis or keywords detection

in this version we review the yml files for general instr test signal and group the detections based on a confidence level label
this detection will be used in our overall repo V2.0

improving the code based on the findings of 

it improve V2.0 to do not detect instru signal only when "test_trigger: gradle via variable hint" as V3.0

segregating flutter integration test which its execution environment is not android style - V4.0

also Updated regex to improve (reduce FP) in detection in cat usage instead of ConnectedAndroidTest




In [1]:
# -*- coding: utf-8 -*-
"""
Scan CI YAML files (plus supporting scripts they invoke) for instrumentation-testing signals and output:
filename, full_name, ci_platform, instru_t_ci_signal, confidence, confidence_reason,
execution_environment, test_invocation, flutter_integ_t_signal, flutter_integ_t_d

Adjustments in this version:
- Treat 'sys-img-*' and other weak hints as NON-evidence unless a strong device signal is also present.
- Remove the fallback that labeled Gradle_Connected when device + any Gradle was present (to avoid false positives).
- Keep renamed invocation taxonomy:
  * Gradle_GMD (managed devices / non-connected *AndroidTest)
  * Gradle_Connected (connected*, Spoon/Marathon/deviceCheck, or explicit Gradle test inputs)
  * Gradle       (BaselineProfile tasks)
  * 3P CLIs, ADB

NEW (this revision):
- Distinguish Third-Party Lab env into 3 categories:
  * Third-Party Lab — Invocation Only
  * Third-Party Lab — Explicit Inline Env
  * Third-Party Lab — Config-Referenced Env
- Produce a sidecar metadata JSONL with provider/env details per file.
"""

import re
import json
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple, Dict, Any, Iterable, Set

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files")
OUTPUT_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV6.0.csv")
OUTPUT_META = OUTPUT_CSV.with_suffix(".meta.jsonl")  # small metadata sidecar

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# === Helpers ===
def extract_full_name_from_file(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    """owner.repo__github++file.yml  -> github"""
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

def build_test_presence_index(tests_dir: Path) -> Set[str]:
    present: Set[str] = set()
    for f in tests_dir.rglob("*"):
        if not f.is_file():
            continue
        name = f.name.lower()
        if "__" in name:
            repo_key = name.split("__", 1)[0]
            present.add(repo_key)
    return present

ANDROIDTEST_PRESENT = build_test_presence_index(TESTS_DIR)

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# Strip comments (#, //, Windows batch comments, PowerShell ::)
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    """
    Normalize YAML blocks so signals in run/script/command are visible:
    - Inline:  "run: ./gradlew ..."    -> "./gradlew ..."
    - Multiline: "run: |" / "run: >"   -> drop key line, keep content
    - Remove YAML list dashes only when they precede likely shell/script lines
    """
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(
        r'(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)',
        r'\1',
        text
    )
    return text

# --- Ignore non-Android infra actions
IGNORE_GHA_ACTIONS_RE = re.compile(
    r'(?mi)^\s*uses\s*:\s*('
    r'docker/(?:setup-qemu-action|setup-buildx-action|build-push-action|login-action)'
    r'|actions/checkout'
    r'|docker/setup-qemu-action'
    r'|docker/setup-buildx-action'
    r')@.*$'
)
def strip_irrelevant_ci_lines(text: str) -> str:
    return IGNORE_GHA_ACTIONS_RE.sub('', text or '')

# Gradle command normalizers
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'
GRADLE_ANYWHERE_RE = re.compile(GRADLE_ANYWHERE)

NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

# Shell-robust prefixes
SHELL_PREFIX = r'(?:\S+=\S+\s+)*(?:sudo\s+)?(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'"]?)?(?:[^#\n;]*?&&\s+)?'

# === Emulator/ADB lines ===
EMULATOR_LINE = (
    rf'(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+'
)
ADB_WAIT_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b'
ADB_SERIAL_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)'

# === Device/Environment sources (NOT triggers) ===
DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [
        r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b'
    ]),

    # Emulator / DIY / Generic runners
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device",   [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@",       [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'\bandroid\b[^\n]*\bcreate\s+avd\b']),
    # CircleCI Android orb hints
    ("Emulator", "circleci android orb", [
        r'(?mi)^\s*(?:-\s*)?android/start-emulator-and-run-tests\s*:',
        r'(?mi)^\s*system-image\s*:\s*system-images;android-\d+;google_apis;'
    ]),

    # Marketplace actions (not DIY)
    ("Emulator", "reactivecircus runner", [r'(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+']),
    ("Emulator", "malinskiy runner", [r'(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+']),

    # sys-img / avdmanager / sdkmanager
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),

    # Weak hints (filtered if no strong signals)
    ("Emulator", "headless flag", [r'(?mi)\b-no-?audio\b', r'(?mi)\b-no-window\b', r'(?mi)\b-no-boot-anim\b']),
    ("Emulator", "avd-name",      [r'(?mi)^\s*avd[-_ ]?name\s*:\s*\S+']),
    ("Emulator", "api-level",     [r'(?mi)\bapi[-_ ]?level\s*:\s*\d{2,}|\bapi_level\s*:\s*\d{2,}']),
    ("Emulator", "abi/arch",      [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image",  [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name",   [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),

    # Third-party labs as environment (presence of CLI/action implies remote lab)
    ("Third_Party_Lab", "gcloud firebase", [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'(?i)\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?mi)^[^\n]*\bmaestro\s+cloud\b']),
    ("Third_Party_Lab", "emulator.wtf action", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),

    # Other emulator actions (not ReactiveCircus/Malinskiy; not 3P labs)
    ("Emulator", "other gha emulator", [
        r'(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+',
        r'(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)'
        r'(?!malinskiy/action-android/emulator-run-cmd@)'
        r'(?!emulator-wtf/run-tests@)'
        r'[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+'
    ]),
]

DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

# === Sanitizers ===
EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")  # strip GHA expressions

def remove_excluded_gradle_tasks(text: str) -> str:
    return EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")

def pre_sanitize(text: str) -> str:
    t = remove_excluded_gradle_tasks(text)
    return GHA_EXPR_RE.sub("", t or "")

# Triggers (primary + anywhere)
TRIGGER_SOURCES_PRIMARY = [
    ("Gradle",  "connectedAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*']),
    ("Gradle", "connected.*Android.*", [
        rf'''(?mix)
        {GRADLE_PREFIX}[^\n\r]*\b
        (?:[:\w-]+:)*                 
        connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b
        (?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)
        [^\n\r]*'''
    ]),
    ("Gradle",  "connectedCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*']),
    ("Gradle",  "cAT shorthand",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*']),
    ("Gradle",  "deviceCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*']),

    # --- GMD invocation
    ("Gradle",  "managedDevice AndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*']),

    # Avoid assemble*AndroidTest
    ("Gradle", "variant/device AndroidTest",
     [rf'''(?mix)
    {GRADLE_PREFIX}[^\n\r]*\b
    (?:
        (?:[:\w-]+:)*              
        (?!{NON_TEST_PREFIX})      
        (?!connected)              
        (?!spoon)                  
        (?!marathon)               
        [A-Za-z0-9][\w-]*androidtest\b  
    )
    [^\n\r]*
    ''']),

    ("Gradle",  "Spoon",    [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle",  "Marathon", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b']),

    ("ADB",     "am instrument", [r'(?mi)^[^\n]*\bam\s+instrument\b']),

    # 3P CLIs (instrumentation-only for FTL)
    ("Third_Party_Lab", "gcloud firebase (instr)", [r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)']),
    ("Third_Party_Lab", "flank",            [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",         [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",    [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "emulator.wtf run", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),
]
TRIGGER_SOURCES_PRIMARY += [
    ("Gradle", "generateBaselineProfile",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*']),
    ("Gradle", "collectBaselineProfile",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*']),
    ("Gradle", "connectedBenchmarkAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*']),
]

TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [
        rf'''(?mix)
        {GRADLE_ANYWHERE}\b
        (?:[:\w-]+:)*                 
        connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b
        '''
    ]),
    ("Gradle", "connectedAndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bconnectedandroidtest\b']),
    ("Gradle", "connectedCheck (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\b(?:[:\w-]+:)*connectedcheck\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)",
     [rf'''(?mix)
    {GRADLE_ANYWHERE}\b
    (?:
        (?:[:\w-]+:)* 
        (?!{NON_TEST_PREFIX})
        (?!connected)
        (?!spoon)
        (?!marathon)
        [A-Za-z0-9][\w-]*androidtest\b
    )
    ''']),
    ("Gradle", "variant/device AndroidTest (anywhere)",
     [rf'''(?mix)
        {GRADLE_ANYWHERE}\b
        (?:
          (?:[:\w-]+:)*
          (?!{NON_TEST_PREFIX})
          [\w-]*androidtest\b
        )
     ''']),
    ("Gradle", "Spoon (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),
]
TRIGGER_SOURCES_ANYWHERE += [
    ("Gradle", "generateBaselineProfile (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bgenerate(?:\w*?)baselineprofile\b']),
    ("Gradle", "collectBaselineProfile (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bcollect(?:\w*?)baselineprofile\b']),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)",
     [rf'(?mi){GRADLE_ANYWHERE}\bconnectedbenchmarkandroidtest\b']),
]

TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

# Inputs to Gradle actions (captures connected/tasks/script under action 'with:' sections)
GHA_GRADLE_INPUTS = compile_any([
    rf'''(?mix)^\s*
      (arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b
      (?:[:\w-]+:)*                
      connected
      (?:\${{\s*[^}}]+\s*}}|[^\n\r])*?
      android
      (?:\${{\s*[^}}]+\s*}}|[^\n\r])*?
      test\b
    ''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    rf'''(?mix)^\s*
      (arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b
      (?:
        (?:[:\w-]+:)*              
        (?!{NON_TEST_PREFIX})      
        [\w-]*androidtest\b
      )
    ''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
])

# GHA inputs that specifically imply GMD (any *AndroidTest NOT starting with connected*)
GHA_GMD_INPUTS = compile_any([
    rf'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b'
    rf'(?!(?:[:\w-]+:)*(?:connected[a-z0-9:._-]*|{NON_TEST_PREFIX})[\w:-]*androidtest\b)'
    rf'(?:[:\w-]+:)*[\w:-]*androidtest\b'
])

# ---------- 3P provider/env/metadata detectors ----------
PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@', r'(?i)\bemulator\.wtf\b'])),
    ("firebase-test-lab", compile_any([r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b', r'(?mi)\bflank\s+android\s+run\b'])),
    ("browserstack", compile_any([r'(?i)\bbrowserstack\b', r'(?i)\bbstack\b'])),
    ("aws-device-farm", compile_any([r'(?mi)\baws\s+devicefarm\b'])),
    ("sauce-labs", compile_any([r'(?mi)\bsaucectl(?:\s+run)?\b', r'(?mi)\bsauce\s+ctl\b'])),
    ("appcenter", compile_any([r'(?mi)\bappcenter\s+test\s+run\s+android\b'])),
    ("maestro-cloud", compile_any([r'(?mi)\bmaestro\s+cloud\b'])),
]

# Inline device / environment hints (explicit in the step)
INLINE_DEVICE_HINTS = compile_any([
    r'(?mi)^\s*devices\s*:\s*\|',
    r'(?mi)\b--device\b',
    r'(?mi)\bmodel\s*=\s*[^,\s]+',
    r'(?mi)\bversion\s*=\s*\d+',
    r'(?mi)\blocale\s*=\s*[-\w]+',
    r'(?mi)\borientation\s*=\s*(portrait|landscape)',
    r'(?mi)^\s*with-orchestrator\s*:\s*true\b',
    r'(?mi)\b--use-orchestrator\b',
    r'(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b',
    r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b',
])

FTL_HAS_INSTRUMENTATION = re.compile(
    r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)'
)
FTL_APP_ONLY = re.compile(
    r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run\b(?![^\n]*\b(--test\b|--type\s+instrumentation\b))'
)

# Config-file references (explicit but not inline)
CONFIG_FILE_HINTS = compile_any([
    r'(?mi)\.ewtf\.ya?ml\b',
    r'(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b',
    r'(?mi)\b--config(?:=|\s+)\S+',
    r'(?mi)\b(browserstack\.ya?ml)\b',
    r'(?mi)\b(bs(?:config)?\.ya?ml)\b',
])

ORCHESTRATOR_HINTS = compile_any([
    r'(?mi)^\s*with-orchestrator\s*:\s*true\b',
    r'(?mi)\b--use-orchestrator\b'
])
RETRY_HINTS = compile_any([
    r'(?mi)\bnum-flaky-test-attempts\s*:\s*(\d+)\b',
    r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)(\d+)\b'
])

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def detect_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def detect_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

def count_devices(text: str) -> int:
    count = 0
    m = re.search(r'(?mi)^\s*devices\s*:\s*\|\s*([\s\S]+)', text)
    if m:
        block = m.group(1)
        lines = [ln for ln in block.splitlines() if ln.strip()]
        pruned = []
        for ln in lines:
            if re.match(r'^\s*\w[\w-]*\s*:\s*', ln):
                break
            pruned.append(ln)
        count += sum(1 for ln in pruned if re.search(r'\bmodel\s*=', ln))
    count += len(re.findall(r'(?mi)\b--device\b', text))
    return count or 0

def detect_orchestrator(text: str) -> bool:
    return any_match(ORCHESTRATOR_HINTS, text)

def detect_retries(text: str) -> int:
    for pat in RETRY_HINTS:
        m = pat.search(text)
        if m:
            try:
                return int(m.group(1))
            except Exception:
                continue
    return 0

# --- Strong/weak classification ---
EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator", "adb -s emulator-serial", "adb wait-for-device",
    "android create avd", "malinskiy runner", "other gha emulator", "circleci android orb",
}
THIRD_PARTY_STRONG_LABELS = {
    "gcloud firebase", "emulator.wtf action", "saucectl",
    "browserstack/bstack", "appcenter test", "maestro cloud"
}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = set()
STRONG_DEVICE_LABELS = EMULATOR_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS | THIRD_PARTY_STRONG_LABELS

def filter_weak_device_hints(labels, groups):
    lbl_set = set(labels)
    if not (lbl_set & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l in STRONG_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

ANDROID_CONTEXT_RE = re.compile(
    r'(?i)\b('
    r'adb|avd|emulator|android\s+sdk|system-images;android-'
    r'|androidtest|connected(check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run'
    r')\b'
)

# --- Flutter helpers (optional) ---
FLUTTER_CMD_LINE_RE = re.compile(r'(?mi)^\s*flutter\s+(?:drive|test)\b[^\n]*')
FLUTTER_DEVICE_FLAG_RE = re.compile(r'(?i)\s+-d\s+(?P<dev>"[^"]+"|\'[^\']+\'|\S+)')
LINUX_HEADLESS_HINTS_RE = re.compile(r'(?mi)^\s*(xvfb-run|export\s+DISPLAY=|sudo\s+Xvfb)\b')
IOS_SIM_HINTS = compile_any([
    r'uses:\s*futureware-tech/simulator-action@',
    r'\bxcrun\s+simctl\b',
    r'\biphonesimulator\b',
    r'\bdestination\b[^\n]*platform=iOS',
    r'(?mi)^\s*model\s*:\s*["\']?\s*(iphone|ipad)\b',
])

# ---------- job splitter ----------
JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw)
    if not m:
        return [("__whole__", raw)]
    jobs_indent = len(m.group("indent"))
    lines = raw.splitlines(True)
    start_idx = raw[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw)]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw)]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        block_text = "".join(lines[i:j])
        blocks.append((name, block_text))
    return blocks

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

def scan_text_for_devices_and_triggers(text: str):
    """Run patterns on arbitrary text block (script or yaml)."""
    sanitized = pre_sanitize(text)

    dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, sanitized.lower())
    dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
    dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)
    trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, sanitized.lower())
    fb_trig_labels, fb_trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, sanitized.lower())
    if fb_trig_labels:
        trig_labels = unique_preserve(trig_labels + fb_trig_labels)
        trig_groups = unique_preserve(trig_groups + fb_trig_groups)
    return dev_labels, dev_groups, trig_labels, trig_groups

# Recognize gradle-build-action as "Gradle present"
GRADLE_BUILD_ACTION_RE = re.compile(r'(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@')

# === Main scan ===
rows: List[Dict[str, Any]] = []
meta_rows: List[Dict[str, Any]] = []  # new: metadata jsonl

for f in sorted(CONFIG_DIR.iterdir()):
    if not f.is_file():
        continue
    if f.suffix.lower() not in (".yml", ".yaml"):
        continue

    filename = f.name
    full_name = extract_full_name_from_file(filename)
    ci_platform = extract_ci_platform(filename)

    try:
        raw = f.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        raw = ""

    file_trigger_labels: List[str] = []
    file_device_labels:  List[str] = []
    file_trigger_groups: List[str] = []
    file_device_groups:  List[str] = []
    file_has_test_trigger = False
    file_has_device_with_group = False
    file_gradle_present_any = False
    file_instru_signal_any = False
    file_flutter_devices: List[str] = []
    file_script_evidence: List[str] = []

    # --- NEW: collect 3P env metadata at file-level (union across jobs)
    file_provider: str = ""
    file_env_declared = False
    file_env_location = "none"   # none | inline | config
    file_device_count = 0
    file_orchestrator = False
    file_retries = 0

    for job_name, job_raw in split_jobs_blocks(raw):
        content = strip_comments(job_raw)
        content = normalize_block_keys(content)
        content = strip_irrelevant_ci_lines(content)
        content_for_triggers = pre_sanitize(content)

        dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, content_for_triggers.lower())
        dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
        dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)

        trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content_for_triggers.lower())

        # ---- GUARD (Item 1): only count GHA gradle inputs when gradle is actually present
        if any_match(GHA_GRADLE_INPUTS, content_for_triggers):
            has_gradle_anywhere = bool(GRADLE_ANYWHERE_RE.search(content_for_triggers) or
                                       GRADLE_BUILD_ACTION_RE.search(content_for_triggers))
            tied_to_emulator_action = bool(re.search(
                r'(?mi)^\s*uses\s*:\s*(?:reactivecircus/android-emulator-runner|'
                r'malinskiy/action-android/emulator-run-cmd|'
                r'hannesa2/action-android/emulator-run-cmd)\@',
                content_for_triggers
            ))
            if has_gradle_anywhere or tied_to_emulator_action:
                trig_labels = unique_preserve(trig_labels + ["gha gradle inputs/script"])
                trig_groups = unique_preserve(trig_groups + ["Gradle"])
        # ---- end guard ----

        if any_match(GHA_GMD_INPUTS, content_for_triggers):
            trig_labels = unique_preserve(trig_labels + ["variant/device AndroidTest"])
            trig_groups = unique_preserve(trig_groups + ["Gradle"])

        fb_trig_labels, fb_trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, content_for_triggers.lower())
        if fb_trig_labels:
            trig_labels = unique_preserve(trig_labels + fb_trig_labels)
            trig_groups = unique_preserve(trig_groups + fb_trig_groups)

        has_test_trigger = bool(trig_labels)

        # --- Guard 2: BrowserStack "upload only" (prevent false positives)
        # If we only saw a Third_Party_Lab label (e.g., browserstack) but there is no
        # test trigger AND no inline devices AND no config refs AND no obvious BS test keywords,
        # drop the Third_Party_Lab evidence.
        if "Third_Party_Lab" in dev_groups:
            inline = detect_inline_env(content_for_triggers)
            cfgref = detect_config_env(content_for_triggers)
            has_ftl_instr = bool(FTL_HAS_INSTRUMENTATION.search(content_for_triggers))
            bs_test_hint = re.search(
                r'(?mi)\b(browserstack|bstack)\b[^\n]*\b(espresso|instrumentation|app-automate|automate|--device|--devices)\b',
                content_for_triggers
            )
            if (not has_test_trigger) and (not inline) and (not cfgref) and (not has_ftl_instr) and (not bs_test_hint):
                # remove generic 3P labels so they don't flip instru_t_ci_signal
                dev_labels = [l for l in dev_labels if l not in {
                    "browserstack/bstack","gcloud firebase","saucectl","appcenter test","maestro cloud","emulator.wtf action"
                }]
                if not any(l in {"browserstack/bstack","gcloud firebase","saucectl","appcenter test","maestro cloud","emulator.wtf action"} for l in dev_labels):
                    dev_groups = [g for g in dev_groups if g != "Third_Party_Lab"]

        # Guard for “other gha emulator” (skip if no Android context & no test trigger)
        android_context = bool(ANDROID_CONTEXT_RE.search(content_for_triggers))
        if ("other gha emulator" in dev_labels) and (not android_context) and (not has_test_trigger):
            dev_labels = [l for l in dev_labels if l != "other gha emulator"]
            if not dev_labels:
                dev_groups = [g for g in dev_groups if g != "Emulator"]

        has_device_setup = bool(dev_labels)
        gradle_present   = bool(GRADLE_ANYWHERE_RE.search(content) or GRADLE_BUILD_ACTION_RE.search(content))

        # Flutter inference (optional)
        if "Flutter" in trig_groups:
            found = []
            for m in FLUTTER_CMD_LINE_RE.finditer(content):
                line = m.group(0)
                d = FLUTTER_DEVICE_FLAG_RE.search(line)
                if d:
                    plat = d.group("dev").strip('"\'')
                    t = plat.lower()
                    if (t == "android" or t.startswith("emulator-") or
                        "sdk gphone" in t or "android sdk built for" in t or "pixel " in t):
                        found.append("android")
                    elif t in {"linux","macos","windows"}:
                        found.append(t)
                    elif t in {"ios","iphone","ipad","iphone simulator"}:
                        found.append("ios")
                    elif t in {"web","web-server","chrome","edge","firefox","safari"}:
                        found.append("web")
            if not found and LINUX_HEADLESS_HINTS_RE.search(content):
                found.append("linux")
            for plat in found:
                if plat and plat not in file_flutter_devices:
                    file_flutter_devices.append(plat)

        has_emulator_group = any(g in {"Emulator", "Third_Party_Lab"} for g in dev_groups)
        has_real_device_strong = any(lbl in REAL_DEVICE_STRONG_LABELS for lbl in dev_labels)

        instru_t_ci_signal_job = bool(
            trig_labels or (has_device_setup and (has_emulator_group or has_real_device_strong))
        )
        file_instru_signal_any = file_instru_signal_any or instru_t_ci_signal_job

        file_device_labels  = unique_preserve(file_device_labels  + dev_labels)
        file_device_groups  = unique_preserve(file_device_groups  + dev_groups)
        file_trigger_labels = unique_preserve(file_trigger_labels + trig_labels)
        file_trigger_groups = unique_preserve(file_trigger_groups + trig_groups)
        file_has_test_trigger = file_has_test_trigger or bool(trig_labels)

        if has_device_setup and (has_emulator_group or has_real_device_strong):
            file_has_device_with_group = True

        file_gradle_present_any = file_gradle_present_any or gradle_present

    # Confidence & reasons
    reasons = []
    if file_has_test_trigger:
        reasons.append("test_trigger: " + ", ".join(file_trigger_labels))
    if file_has_device_with_group:
        msg = "device_setup: " + ", ".join(file_device_labels) if file_device_labels else "device_setup"
        if file_gradle_present_any:
            msg += "; gradle present"
        if file_script_evidence:
            msg += f"; scripts: {', '.join(file_script_evidence)}"
        reasons.append(msg)
    reason = " | ".join(r for r in reasons if r)

    confidence = ""
    if file_has_test_trigger and file_has_device_with_group:
        confidence = "high"
    elif file_has_test_trigger or file_has_device_with_group:
        confidence = "medium"

    # AndroidTest presence boost
    if file_instru_signal_any and full_name in ANDROIDTEST_PRESENT:
        if confidence == "":
            confidence = "medium"
        elif confidence == "medium":
            confidence = "high"
        reason = (reason + " + boosted (AndroidTest present)").strip()

    # === Multi-valued test_invocation ===
    def has_gmd_gradle_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        if any("manageddevice androidtest" in l for l in L):
            return True
        if ("variant/device androidtest" in L
            and not any(x in L for x in {
                "connected.*android.*",
                "connectedandroidtest",
                "connectedbenchmarkandroidtest",
                "spoon",
                "marathon",
            })):
            return True
        return False

    def has_connected_gradle_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        keys = {
            "connected.*android.*",
            "connectedandroidtest",
            "connectedbenchmarkandroidtest",
            "connectedcheck",
            "cat shorthand",
            "connected (anywhere)",
            "connectedandroidtest (anywhere)",
            "connectedcheck (anywhere)",
            "spoon",
            "marathon",
            "devicecheck",
            "gha gradle inputs/script",
        }
        if any(k in L for k in keys):
            return True
        return any(("connected" in l and "android" in l and "test" in l) for l in L)

    def has_baselineprofile_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        return any("baselineprofile" in l for l in L)

    def map_test_invocations(groups: List[str], trigger_labels: List[str]) -> List[str]:
        s_groups = set(groups)
        out: List[str] = []
        if has_gmd_gradle_trigger(trigger_labels):
            out.append("Gradle_GMD")
        if "Third_Party_Lab" in s_groups:
            out.append("3P CLIs")
        if "ADB" in s_groups:
            out.append("ADB")
        if has_connected_gradle_trigger(trigger_labels):
            out.append("Gradle_Connected")
        if has_baselineprofile_trigger(trigger_labels):
            out.append("Gradle")
        return sorted(set(out))

    def map_execution_envs(groups: List[str], labels: List[str]) -> List[str]:
        envs = set()
        s_groups, s_labels = set(groups), set(labels)

        if "Third_Party_Lab" in s_groups:
            envs.add("Third Party")
        if "Real_Device" in s_groups:
            envs.add("Real Device")

        if "Emulator" in s_groups:
            has_reactivecircus = "reactivecircus runner" in s_labels
            has_malinskiy     = "malinskiy runner" in s_labels
            has_other_action  = "other gha emulator" in s_labels

            if has_reactivecircus or has_malinskiy or has_other_action:
                envs.add("Emulator_Generic")
                if has_reactivecircus:
                    envs.add("Emulator_ReactiveCircus")
                if has_malinskiy:
                    envs.add("Emulator_Malinskiy")
                if has_other_action:
                    envs.add("Emulator_Other")
            else:
                envs.add("Emulator_DIY")

        return sorted(envs)

    # --- 3P label into 3 buckets
    third_party_label = ""
    if "Third Party" in map_execution_envs(file_device_groups, file_device_labels):
        if file_env_declared and file_env_location == "inline":
            third_party_label = "Third-Party Lab — Explicit Inline Env"
        elif file_env_declared and file_env_location == "config":
            third_party_label = "Third-Party Lab — Config-Referenced Env"
        else:
            third_party_label = "Third-Party Lab — Invocation Only"

    combined_groups = unique_preserve(file_trigger_groups + file_device_groups)
    # core CSV row
    csv_row = {
        "filename": filename,
        "full_name": full_name,
        "ci_platform": ci_platform,
        "instru_t_ci_signal": bool(file_instru_signal_any),
        "confidence": confidence if file_instru_signal_any else "",
        "confidence_reason": reason if file_instru_signal_any else "",
        "execution_environment": ",".join(map_execution_envs(file_device_groups, file_device_labels)),
        "test_invocation": ",".join(map_test_invocations(combined_groups, file_trigger_labels)),
        "flutter_integ_t_signal": ("Flutter" in file_trigger_groups),
        "flutter_integ_t_d": ",".join(sorted(set(file_flutter_devices))),
        "third_party_env_label": third_party_label,
    }
    rows.append(csv_row)

    # metadata sidecar row (JSONL)
    meta_rows.append({
        "filename": filename,
        "full_name": full_name,
        "provider": file_provider or "other",
        "envDeclared": bool(file_env_declared),
        "envLocation": file_env_location,   # none | inline | config
        "deviceCount": int(file_device_count),
        "usesOrchestrator": bool(file_orchestrator),
        "retries": int(file_retries),
    })

# Save CSV
out_df = pd.DataFrame(rows)
out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# Save metadata JSONL
with OUTPUT_META.open("w", encoding="utf-8") as jf:
    for m in meta_rows:
        jf.write(json.dumps(m, ensure_ascii=False) + "\n")

print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")
print(f"Saved metadata: {OUTPUT_META} (rows={len(meta_rows)})")


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV6.0.csv (rows=12667)
Saved metadata: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_YML_FilesV6.0.meta.jsonl (rows=12667)


Instru Testing Signals from Build / Config files

In [7]:
# -*- coding: utf-8 -*-
"""
Instru Test Signal Build / Config — Hybrid (V6.0)

Goals:
- High recall + good precision by combining:
  • Robust, brace-balanced discovery of GMD (managed devices) blocks, and
  • Broader V5-style instrumentation config signals (deps/runner/plugin/gating).
- Scan ALL Gradle scripts (*.gradle, *.gradle.kts), not just build.* files.
- Be resilient to nested blocks inside testOptions{...}. No fragile [^}] patterns.
- Extract literals when present (device/apiLevel/systemImageSource/identifier).
- Emit one row per file (so you can see coverage), adding GMD spans as structured rows.

Notes:
- We prune noisy directories but still traverse conventional Gradle locations
  (root, modules, buildSrc, gradle/, scripts/, included builds, etc.).
- We prefilter tokens before deep parsing for speed, but still emit a file-level
  row even when nothing is detected, to remain inclusive/traceable.

Adds (vs. earlier versions):
- GMD device name detection for both simple and FQCN types, with optional ::class.
- GMD create("name", ManagedVirtualDevice[::class]) forms (KTS/Groovy).
- OPTIONAL: generic create<ManagedVirtualDevice>("name") forms (KTS).
- Property setters (.set(...)) and single-quoted / unquoted values.
- Expanded inner-token detection (deviceGroups).
- Slightly wider prefilter token set.
- Extra instrumentation cues (androidTestUtil, orchestrator execution).
"""

import os
import re
import pandas as pd
from typing import List, Tuple, Optional, Dict, Iterable

# === CONFIG ===
ROOT    = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files"
OUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_Instru_T_Signal_ConfigV6.0.csv"

# --- Ignore folders commonly not relevant ---
SKIP_DIRS = {".git", ".idea", ".gradle", "build", "out", "node_modules", ".github", ".gitlab"}

# --- File filters ---
def is_any_gradle_file(path: str) -> bool:
    p = path.lower()
    return p.endswith(".gradle") or p.endswith(".gradle.kts")

# --- Repo key extraction (owner.repo) ---
def extract_full_name(path: str) -> str:
    """
    Best-effort to recover owner.repo, given your 'All_Config_Files' naming.
    Adjust if your layout encodes the repo differently.
    """
    base = os.path.basename(path).lower()
    if "__" in base:
        return base.split("__", 1)[0]
    return os.path.basename(os.path.dirname(path)).lower()

# --- Comment stripping (handles // and /* */ but preserves http(s)://) ---
def strip_comments_gradle(text: str) -> str:
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.S)            # block comments
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.M)            # line comments (not http://)
    return s

# --- Balanced block helper ---
def _find_block_span_from_head(text: str, head_start: int) -> Optional[Tuple[int, int]]:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

# convert char indexes to (start_line, end_line)
def _char_to_line_span(text: str, ci: Tuple[int, int]) -> Tuple[int, int]:
    a, b = ci
    start_line = text[:a].count("\n") + 1
    end_line   = text[:b+1].count("\n") + 1
    return start_line, end_line

# --- GMD field/name extractors ---
# Accept foo = "...", foo = '...', foo = identifier, and foo.set("...") / foo.set('...')
GMD_DEVICE_RX = re.compile(
    r'\bdevice\s*=\s*(?:"([^"]+)"|\'([^\']+)\'|([A-Za-z0-9_.]+))'
    r'|\bdevice\.set\(\s*(?:"([^"]+)"|\'([^\']+)\')\s*\)',
    re.I
)
GMD_API_RX = re.compile(
    r'\bapiLevel\s*=\s*"?(\d+)"?'
    r'|\bapiLevel\.set\(\s*"?(\d+)"?\s*\)',
    re.I
)
# OPTIONAL adjustment: allow unquoted identifiers for systemImageSource too
GMD_SRC_RX = re.compile(
    r'\bsystemImageSource\s*=\s*(?:"([^"]+)"|\'([^\']+)\'|([A-Za-z0-9_.-]+))'
    r'|\bsystemImageSource\.set\(\s*(?:"([^"]+)"|\'([^\']+)\')\s*\)',
    re.I
)

# Kotlin DSL: myPixel(ManagedVirtualDevice) { ... }   or fully qualified or ::class
GMD_BLOCK_NAME_RX = re.compile(
    r'(\w+)\s*\(\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
    re.I
)
# Also support create("pixel6Api34", ManagedVirtualDevice[::class]) { ... }
GMD_CREATE_RX = re.compile(
    r'create\(\s*"([^"]+)"\s*,\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice(?:\s*::\s*class)?\s*\)',
    re.I
)
# OPTIONAL: KTS generic create<ManagedVirtualDevice>("name")
GMD_CREATE_GENERIC_RX = re.compile(
    r'create\s*<\s*(?:com\.android\.build\.api\.dsl\.)?ManagedVirtualDevice\s*>\s*\(\s*"([^"]+)"\s*\)',
    re.I
)

# --- Token sets for quick prefiltering ---
PREFILTER_TOKENS = {
    # GMD-ish
    "manageddevices", "manageddevice", "managedvirtualdevice", "devicegroups", "devices", "groups", "testoptions",
    "alldevicesandroidtest", "pixel", "apilevel", "systemimagesource",
    # V5-style broader signals
    "androidtestimplementation", "androidtestapi", "androidtestruntimeonly", "androidtestcompileonly", "androidtestcompile",
    "testinstrumentationrunner", "testinstrumentationrunnerarguments", "enableandroidtest", "com.android.test",
    "androidtestutil", "androidx_test_orchestrator", "orchestrator"
}

# --- Find GMD blocks using a two-phase balanced approach ---
TESTOPTIONS_HEAD = re.compile(r"\btestOptions\s*\{", re.I)
MANAGED_HEAD     = re.compile(r"\bmanagedDevices\s*\{", re.I)

# Include deviceGroups explicitly
GMD_INNER_TOKENS_RX = re.compile(r"\b(managedDevices|managedVirtualDevice|devices|groups|deviceGroups)\b", re.I)

def find_gmd_line_spans(text: str) -> List[Tuple[int, int]]:
    """Return a list of (start_line, end_line) for candidate GMD-containing blocks."""
    spans: List[Tuple[int, int]] = []

    # 1) Direct managedDevices { ... } blocks
    for m in MANAGED_HEAD.finditer(text):
        ci = _find_block_span_from_head(text, m.start())
        if ci:
            spans.append(_char_to_line_span(text, ci))

    # 2) testOptions { ... } blocks that contain GMD tokens anywhere inside
    for m in TESTOPTIONS_HEAD.finditer(text):
        ci = _find_block_span_from_head(text, m.start())
        if not ci:
            continue
        a, b = ci
        sub = text[a:b+1]
        if GMD_INNER_TOKENS_RX.search(sub):
            spans.append(_char_to_line_span(text, (a, b)))

    return spans

def _first_group(m: re.Match) -> Optional[str]:
    if not m:
        return None
    for g in m.groups():
        if g:
            return g
    return None

def parse_gmd_block(text: str, span: Tuple[int, int]) -> Dict[str, Optional[str]]:
    lines = (text or "").splitlines()
    block_text = "\n".join(lines[span[0]-1:span[1]])
    norm: Dict[str, Optional[str]] = {
        "device_profile": None,
        "api_level": None,
        "image_source": None,
        "device_identifier": None,
        "context_anchor": None,
    }
    if (m := GMD_DEVICE_RX.search(block_text)):
        norm["device_profile"] = _first_group(m)
    if (m := GMD_API_RX.search(block_text)):
        norm["api_level"] = _first_group(m)
    if (m := GMD_SRC_RX.search(block_text)):
        norm["image_source"] = _first_group(m)
    if (m := GMD_BLOCK_NAME_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    elif (m := GMD_CREATE_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    elif (m := GMD_CREATE_GENERIC_RX.search(block_text)):
        ident = m.group(1)
        norm["device_identifier"] = ident
        norm["context_anchor"] = f"managedDevices.{ident}"
    return norm

# --- V5-style broader instrumentation signals ---
V5_SIGNAL_TOKENS = [
    "androidtestimplementation", "androidtestapi", "androidtestcompileonly",
    "androidtestruntimeonly", "androidtestcompile",
    "testinstrumentationrunner", "testinstrumentationrunnerarguments",
    "androidtestutil",                                # orchestrator artifact often present
    'execution "androidx_test_orchestrator"',         # Groovy
    "execution 'androidx_test_orchestrator'",         # Groovy
    'execution("ANDROIDX_TEST_ORCHESTRATOR")',        # KTS (rare)
    "enableandroidtest",
    'id("com.android.test")', "id \'com.android.test\'", 'apply plugin: "com.android.test"'
]

def has_instru_signal_config(text: str) -> bool:
    t = (text or "").lower()
    return any(tok in t for tok in V5_SIGNAL_TOKENS)

# --- Walk & scan ---
def walk_gradle_files(root: str) -> List[str]:
    out: List[str] = []
    for dirpath, dirnames, filenames in os.walk(root):
        # prune noisy dirs
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for fname in filenames:
            if is_any_gradle_file(fname):
                out.append(os.path.join(dirpath, fname))
    return out

def prefilter_interesting(text: str) -> bool:
    t = (text or "").lower()
    return any(tok in t for tok in PREFILTER_TOKENS)

def scan_file(path: str) -> Dict[str, Iterable[Dict[str, Optional[str]]]]:
    """Return dict with keys: text, v5_signal(bool), gmd_spans(list[span tuples]), gmd_detections(list[dict])."""
    try:
        raw = open(path, "r", encoding="utf-8", errors="ignore").read()
    except Exception:
        raw = ""
    text = strip_comments_gradle(raw)

    v5_signal = has_instru_signal_config(text)

    gmd_spans: List[Tuple[int, int]] = []
    gmd_detections: List[Dict[str, Optional[str]]] = []

    if prefilter_interesting(text):
        gmd_spans = find_gmd_line_spans(text)
        for sp in gmd_spans:
            det = parse_gmd_block(text, sp)
            det["span_start_line"], det["span_end_line"] = sp
            gmd_detections.append(det)

    return {
        "text": text,
        "v5_signal": v5_signal,
        "gmd_spans": gmd_spans,
        "gmd_detections": gmd_detections,
    }

# --- Main ---
def main():
    rows: List[Dict[str, Optional[str]]] = []
    files = walk_gradle_files(ROOT)

    for f in sorted(files):
        full = extract_full_name(f)
        result = scan_file(f)
        v5_signal = bool(result["v5_signal"])  # broader instrumentation signals
        gmd_detections = list(result["gmd_detections"])  # list of dicts

        if gmd_detections:
            for d in gmd_detections:
                rows.append({
                    "full_name": full,
                    "filename": os.path.relpath(f, ROOT),
                    "instru_t_signal_config": v5_signal,
                    "gmd_present": True,
                    "device_profile": d.get("device_profile"),
                    "api_level": d.get("api_level"),
                    "image_source": d.get("image_source"),
                    "device_identifier": d.get("device_identifier"),
                    "context_anchor": d.get("context_anchor"),
                    "span_start_line": d.get("span_start_line"),
                    "span_end_line": d.get("span_end_line"),
                })
        else:
            # Include a file-level row even with no GMD spans found
            rows.append({
                "full_name": full,
                "filename": os.path.relpath(f, ROOT),
                "instru_t_signal_config": v5_signal,
                "gmd_present": False,
                "device_profile": None,
                "api_level": None,
                "image_source": None,
                "device_identifier": None,
                "context_anchor": None,
                "span_start_line": None,
                "span_end_line": None,
            })

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUT_CSV} (rows={len(rows)})")

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.1.1_Instru_T_Signal_ConfigV6.0.csv (rows=29478)
